In [10]:
import cv2
import numpy as np
import pandas as pd
import os, glob
from pathlib import Path

In [11]:
BASE_DIR     = r"D:\program vscode\MoneyLens\ai\Dataset_ocr"
SPLITS       = ["train", "valid", "test"]
IMG_H, IMG_W = 32, 128
CHANNELS     = 1

In [12]:
LABEL_CLASSES = [
    "harga_satuan",
    "nama_produk",
    "QTY",
    "tanggal",
    "total_harga_barang",
    "total_transaksi",
]

In [13]:
def load_annotations(split: str) -> pd.DataFrame:
    csv_path = os.path.join(BASE_DIR, split, "_annotations.csv")
    if not os.path.exists(csv_path):
        print(f"  [WARNING] Tidak ada annotations: {csv_path}")
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    print(f"  Annotations {split}: {len(df)} baris")
    print(f"  Kelas tersedia: {df['class'].unique().tolist()}")
    return df

In [14]:
MIN_CROP_size = 5
PADDING       = 6

def add_padding(crop_cv: np.ndarray,
                img_cv: np.ndarray,
                x1: int, y1: int,
                x2: int, y2: int) -> np.ndarray:
    """
    Tambah padding di tiap sisi crop dengan mengambil
    area lebih luas dari gambar asli.
    Ini mencegah teks terpotong saat crop terlalu mepet.
    """
    img_h, img_w = img_cv.shape[:2]
    x1p = max(0,     x1 - PADDING)
    y1p = max(0,     y1 - PADDING)
    x2p = min(img_w, x2 + PADDING)
    y2p = min(img_h, y2 + PADDING)
    return img_cv[y1p:y2p, x1p:x2p]

def preprocess_crop(crop_cv: np.ndarray) -> np.ndarray:
    """
    Preprocessing untuk 1 crop region teks.
 
    Alur:
      1. Grayscale       → sederhanakan info visual
      2. Denoise         → kurangi noise foto struk
      3. CLAHE           → normalisasi pencahayaan lokal
      4. Adaptive Thresh → binarisasi bersih
      5. Resize 128x32   → sesuaikan ukuran input model
      6. Normalisasi     → pixel [0.0, 1.0]
 
    Returns:
      np.ndarray shape (32, 128, 1), dtype float32
    """
    # 1. Cek ukuran minimum
    h, w = crop_cv.shape[:2]
    if h < MIN_CROP_SIZE or w < MIN_CROP_SIZE:
        raise ValueError(
            f"Crop terlalu kecil ({w}x{h}px), minimum {MIN_CROP_SIZE}px"
        )
    
    # 2. Grayscale
    if len(crop_cv.shape) == 3:
        gray = cv2.cvtColor(crop_cv, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop_cv
 
    # 3. Denoise
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
 
    # 4. CLAHE — normalisasi pencahayaan tidak merata
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
 
    # 5. Adaptive Threshold — binarisasi
    block = max(3, min(h,w) //2 * 2 + 1)
    binary = cv2.adaptiveThreshold(
        enhanced, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, block, 10
    )
 
    # 6. Resize ke ukuran input model
    resized = cv2.resize(binary, (IMG_W, IMG_H),
                         interpolation=cv2.INTER_AREA)
 
    # 7. Normalisasi [0.0, 1.0] + tambah channel dimension
    normalized = resized.astype(np.float32) / 255.0
    return normalized.reshape(IMG_H, IMG_W, CHANNELS)

In [15]:
def crop_by_class(img_cv: np.ndarray,
                  df_ann: pd.DataFrame,
                  filename: str) -> dict:
    """
    Crop tiap region teks berdasarkan kelas dari annotations CSV.
 
    Returns:
      dict { class_name: [crop_array, ...] }
    """
    result = {cls: [] for cls in LABEL_CLASSES}
 
    rows = df_ann[df_ann["filename"] == filename]
    for _, row in rows.iterrows():
        cls = str(row.get("class", ""))
        if cls not in LABEL_CLASSES:
            continue
        try:
            x1 = max(0, int(row["xmin"]))
            y1 = max(0, int(row["ymin"]))
            x2 = min(img_cv.shape[1], int(row["xmax"]))
            y2 = min(img_cv.shape[0], int(row["ymax"]))
 
            if x2 <= x1 or y2 <= y1:
                continue
 
            crop = add_padding(img_cv, img_cv, x1, y1, x2, y2)
            if crop.size > 0:
                result[cls].append(crop)
        except Exception:
            continue
 
    return result

In [16]:
def process_image(img_path: str,
                  df_ann: pd.DataFrame,
                  crops_dir: str,
                  arrays_dir: str) -> dict:
    """
    Proses 1 gambar struk:
      - Crop tiap region per kelas
      - Preprocessing tiap crop
      - Simpan PNG (visual) + NPY (input model)
 
    Returns:
      dict ringkasan crop per kelas
    """
    img_cv = cv2.imread(img_path)
    if img_cv is None:
        raise ValueError(f"Tidak bisa buka: {img_path}")
 
    fname   = Path(img_path).name
    stem    = Path(img_path).stem
    h, w    = img_cv.shape[:2]
 
    # Crop per kelas
    crops_by_class = crop_by_class(img_cv, df_ann, fname)
 
    summary = {}
    for cls, crops in crops_by_class.items():
        summary[cls] = len(crops)
        for i, crop in enumerate(crops):
            try:
                # Preprocessing
                arr = preprocess_crop(crop)
 
                # Simpan PNG untuk cek visual
                png_name = f"{stem}_{cls}_{i:02d}.png"
                cv2.imwrite(
                    os.path.join(crops_dir, png_name),
                    (arr[:, :, 0] * 255).astype(np.uint8)
                )
 
                # Simpan NPY untuk input model
                npy_name = f"{stem}_{cls}_{i:02d}.npy"
                np.save(os.path.join(arrays_dir, npy_name), arr)
 
            except Exception as e:
                print(f"    [ERROR] {cls} crop {i}: {e}")
 
    return summary

In [17]:
print("=" * 60)
print("TASK 2: PREPROCESSING FORMAT INPUT MODEL OCR")
print("=" * 60)
print(f"Target size  : {IMG_H}x{IMG_W}px (HxW)")
print(f"Output format: float32 array shape ({IMG_H},{IMG_W},{CHANNELS})")
print(f"Kelas        : {LABEL_CLASSES}")
print()
 
grand_total = {cls: 0 for cls in LABEL_CLASSES}
 
for split in SPLITS:
    img_dir    = os.path.join(BASE_DIR, split)
    crops_dir  = os.path.join(BASE_DIR, "preprocessed", split, "crops")
    arrays_dir = os.path.join(BASE_DIR, "preprocessed", split, "arrays")
    os.makedirs(crops_dir,  exist_ok=True)
    os.makedirs(arrays_dir, exist_ok=True)
 
    df_ann    = load_annotations(split)
    if df_ann.empty:
        continue
 
    img_paths = sorted(
        glob.glob(os.path.join(img_dir, "*.jpg")) +
        glob.glob(os.path.join(img_dir, "*.png"))
    )
 
    print(f"\n[{split.upper()}] {len(img_paths)} gambar")
    print(f"  Output crops  : {crops_dir}")
    print(f"  Output arrays : {arrays_dir}")
 
    split_total = {cls: 0 for cls in LABEL_CLASSES}
    ok, fail    = 0, 0
 
    for i, path in enumerate(img_paths, 1):
        try:
            summary = process_image(path, df_ann, crops_dir, arrays_dir)
            for cls, n in summary.items():
                split_total[cls]  += n
                grand_total[cls]  += n
            ok += 1
            print(f"  [{i:03d}/{len(img_paths):03d}] ✓ {Path(path).name} "
                  f"| {dict((k,v) for k,v in summary.items() if v > 0)}")
        except Exception as e:
            print(f"  [{i:03d}/{len(img_paths):03d}] ✗ {Path(path).name} → {e}")
            fail += 1
 
    print(f"\n  Ringkasan {split}:")
    for cls, n in split_total.items():
        print(f"    {cls:<22}: {n:>4} crop")
    print(f"  Berhasil: {ok} | Gagal: {fail}")

TASK 2: PREPROCESSING FORMAT INPUT MODEL OCR
Target size  : 32x128px (HxW)
Output format: float32 array shape (32,128,1)
Kelas        : ['harga_satuan', 'nama_produk', 'QTY', 'tanggal', 'total_harga_barang', 'total_transaksi']

  Annotations train: 3194 baris
  Kelas tersedia: ['nama_produk', 'QTY', 'harga_satuan', 'total_harga_barang', 'total_transaksi', 'tanggal']

[TRAIN] 283 gambar
  Output crops  : D:\program vscode\MoneyLens\ai\Dataset_ocr\preprocessed\train\crops
  Output arrays : D:\program vscode\MoneyLens\ai\Dataset_ocr\preprocessed\train\arrays
    [ERROR] harga_satuan crop 0: name 'MIN_CROP_SIZE' is not defined
    [ERROR] harga_satuan crop 1: name 'MIN_CROP_SIZE' is not defined
    [ERROR] nama_produk crop 0: name 'MIN_CROP_SIZE' is not defined
    [ERROR] nama_produk crop 1: name 'MIN_CROP_SIZE' is not defined
    [ERROR] QTY crop 0: name 'MIN_CROP_SIZE' is not defined
    [ERROR] QTY crop 1: name 'MIN_CROP_SIZE' is not defined
    [ERROR] tanggal crop 0: name 'MIN_CROP_S

In [18]:
print(f"\n{'='*60}")
print("TOTAL CROP PER KELAS (semua split)")
print(f"{'='*60}")
for cls, n in grand_total.items():
    prio = " ← PRIORITAS" if cls in ["total_transaksi", "tanggal"] else ""
    print(f"  {cls:<22}: {n:>5} array .npy{prio}")
 
print(f"\nOutput siap digunakan untuk Task 1 & Task 3")
print(f"Lokasi: {os.path.join(BASE_DIR, 'preprocessed')}")
print(f"{'='*60}")


TOTAL CROP PER KELAS (semua split)
  harga_satuan          :   739 array .npy
  nama_produk           :  1031 array .npy
  QTY                   :   950 array .npy
  tanggal               :   383 array .npy ← PRIORITAS
  total_harga_barang    :  1027 array .npy
  total_transaksi       :   401 array .npy ← PRIORITAS

Output siap digunakan untuk Task 1 & Task 3
Lokasi: D:\program vscode\MoneyLens\ai\Dataset_ocr\preprocessed
